# 01 - DINO Classification and Embeddings

In this notebook we use DINO embeddings as frozen features and train a lightweight classifier.

## Why this matters for defect detection
- You can build strong baselines quickly with limited labels.
- Embedding space analysis helps detect unusual package/device appearances.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
ARTIFACTS_DIR = ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


## 1) Load DINO backbone
If DINOv3 is available in your runtime, this code can be extended by adding candidate names.


In [ ]:
model_name = 'vit_base_patch14_dinov2.lvd142m'
available = set(timm.list_models(pretrained=True))
if model_name not in available:
    model_name = 'vit_small_patch14_dinov2.lvd142m'

model = timm.create_model(model_name, pretrained=True, num_classes=0)
model.eval().to(device)
print('Loaded:', model_name)


## 2) Prepare CIFAR-10 subset
We use a subset to keep runtime fast during your flight.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds_full = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_ds_full = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=transform)

train_idx = np.arange(0, 5000)
test_idx = np.arange(0, 1000)

train_ds = Subset(train_ds_full, train_idx)
test_ds = Subset(test_ds_full, test_idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

class_names = train_ds_full.classes
print('Classes:', class_names)
print('Train subset:', len(train_ds), 'Test subset:', len(test_ds))


## 3) Feature extraction helper
This converts images into fixed DINO embeddings.


In [ ]:
def extract_embeddings(loader, model, device='cpu'):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = model(x)
            feats.append(f.detach().cpu().numpy())
            labels.append(y.numpy())
    X = np.concatenate(feats, axis=0)
    Y = np.concatenate(labels, axis=0)
    return X, Y

X_train, y_train = extract_embeddings(train_loader, model, device=device)
X_test, y_test = extract_embeddings(test_loader, model, device=device)

print('X_train:', X_train.shape, 'X_test:', X_test.shape)


## 4) Train linear classifier
Classic transfer learning baseline: frozen backbone + linear model.


In [ ]:
clf = LogisticRegression(max_iter=2000, n_jobs=-1, verbose=0)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

acc = accuracy_score(y_test, pred)
print('Test accuracy:', round(acc, 4))
print(classification_report(y_test, pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (DINO Embeddings + Logistic Regression)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


## 5) Nearest-neighbor retrieval demo
This is useful for visual QA and anomaly exploration.


In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=6, metric='cosine')
nn.fit(X_train)

query_idx = 5
dist, idxs = nn.kneighbors(X_test[query_idx:query_idx+1], return_distance=True)
print('Query label:', class_names[y_test[query_idx]])
print('Neighbor labels:', [class_names[y_train[i]] for i in idxs[0]])
print('Distances:', dist[0])


## 6) Save artifacts
These artifacts can be reused in deployment prototypes.


In [ ]:
import joblib

joblib.dump(clf, ARTIFACTS_DIR / 'linear_classifier_dino.pkl')
np.save(ARTIFACTS_DIR / 'X_train_embeddings.npy', X_train)
np.save(ARTIFACTS_DIR / 'y_train_labels.npy', y_train)
print('Saved artifacts to:', ARTIFACTS_DIR)


## 7) Transition to defect detection mindset

Replace CIFAR images with your logistics images and labels:
- `normal_box`, `crushed_corner`, `seal_broken`, `screen_crack_suspected`
- Keep the same embedding extraction and linear baseline pattern.

Next: detection, segmentation, and video stream logic in notebook 02.
